In [ ]:
import pandas as pd
import numpy as np 
s1 = pd.read_csv("ML2026/cleaned dataset/source1.tsv", sep='\t')
s2 = pd.read_csv("ML2026/cleaned dataset/source2.tsv", sep='\t')
s3 = pd.read_csv("ML2026/cleaned dataset/source3.tsv", sep='\t')

<h1> Method1. Block 1: exact normalized name + Transliterated name + Token based name blocking + country based + Common token removal</h1>

In [ ]:
s1 = s1.reset_index(drop=True)

s1["s1_idx"] = np.arange(len(s1), dtype=np.int32)
# s1.iloc[candidate_idx]["entity_id"] use this to fetch from s1

In [ ]:
from collections import defaultdict, Counter

# Token frequency
token_freq = Counter()

for name in s2["norm_name"].dropna():
    token_freq.update(set(name.split()))

MAX_TOKEN_FREQ = 5000

# Country + token index
country_token_index = defaultdict(list)

for idx, name, country in zip(
    s2.index,
    s2["norm_name"],
    s2["country"]
):
    if not name or not country:
        continue

    for token in set(name.split()):
        if len(token) >= 3 and token_freq[token] <= MAX_TOKEN_FREQ:
            country_token_index[(country, token)].append(idx)


# Country + exact normalized name
name_index = defaultdict(list)

for idx, name, country in zip(
    s2.index,
    s2["norm_name"],
    s2["country"]
):
    if name and country:
        name_index[(country, name)].append(idx)


# Country + exact transliterated name
trans_index = defaultdict(list)

for idx, name, country in zip(
    s2.index,
    s2["Trans_name"],
    s2["country"]
):
    if name and country:
        trans_index[(country, name)].append(idx)


def generate_candidates(row):
    candidates = set()
    country = row["country"]

    if not country:
        return []

    # Exact normalized name
    if row["norm_name"]:
        candidates.update(
            name_index.get((country, row["norm_name"]), [])
        )

    # Exact transliterated name
    if row["Trans_name"]:
        candidates.update(
            trans_index.get((country, row["Trans_name"]), [])
        )

    # Token blocking
    if row["norm_name"]:
        for token in set(row["norm_name"].split()):
            if len(token) >= 3 and token_freq[token] <= MAX_TOKEN_FREQ:
                candidates.update(
                    country_token_index.get((country, token), [])
                )

    return list(candidates)

In [ ]:
df = s1.sample(100000)

In [ ]:
candidate_counts = df.apply(
    lambda row: len(generate_candidates(row)),
    axis=1
)

print(candidate_counts.describe())
print("0 candidates:", (candidate_counts == 0).sum())
print("1 candidate:", (candidate_counts == 1).sum())
print(">1 candidates:", (candidate_counts > 1).sum())

# METHOD2: using exact&Transliterated names + Phonetics + TF-IDF name token matching + TF-IDF address token matching 

In [ ]:
!pip install jellyfish

In [ ]:
import pandas as pd
from collections import defaultdict, Counter
import jellyfish
import gc
import csv
class StrictCandidateGenerator:
    # Drastically reduced thresholds
    def __init__(self, max_name_freq=100, max_addr_freq=50):
        self.max_name_freq = max_name_freq
        self.max_addr_freq = max_addr_freq
        
        self.name_token_freq = Counter()
        self.addr_token_freq = Counter()
        
        self.exact_name_idx = defaultdict(list)
        self.trans_name_idx = defaultdict(list)
        self.phonetic_idx = defaultdict(list)
        self.name_token_idx = defaultdict(list)
        self.addr_token_idx = defaultdict(list)

    def _get_val(self, val):
        return val if pd.notna(val) and str(val).strip() else ""

    def fit(self, df):
        print("Calculating token frequencies...")
        for row in df.itertuples(index=False):
            name = self._get_val(row.clean_name)
            addr = self._get_val(row.norm_address)
            
            if name:
                self.name_token_freq.update(name.split())
            if addr:
                self.addr_token_freq.update(addr.split())

        print("Building strict indices...")
        for row in df.itertuples(index=True):
            idx = row.Index 
            country = self._get_val(row.country) or "UNKNOWN"
            name = self._get_val(row.clean_name)
            trans = self._get_val(row.Trans_name)
            addr = self._get_val(row.norm_address)

            if name:
                self.exact_name_idx[(country, name)].append(idx)
                
                # Strict Phonetic: Only hash names 5 chars or longer
                if len(name) >= 5:
                    phonetic_key = jellyfish.metaphone(name)[:6]
                    self.phonetic_idx[(country, phonetic_key)].append(idx)
                
                # Strict Tokens: Minimum 4 chars, heavily capped frequency
                for token in set(name.split()):
                    if len(token) >= 4 and self.name_token_freq[token] <= self.max_name_freq:
                        self.name_token_idx[(country, token)].append(idx)

            if trans:
                self.trans_name_idx[(country, trans)].append(idx)

            if addr and country != "UNKNOWN":
                for token in set(addr.split()):
                    if len(token) >= 5 and self.addr_token_freq.get(token, 0) <= self.max_addr_freq:
                        self.addr_token_idx[(country, token)].append(idx)

    def get_candidates(self, query_row):
        candidates = set()
        
        country = self._get_val(query_row.get("country")) or "UNKNOWN"
        name = self._get_val(query_row.get("clean_name"))
        trans = self._get_val(query_row.get("Trans_name"))
        addr = self._get_val(query_row.get("norm_address"))

        # 1. Exact Matches (Always allowed)
        if name:
            candidates.update(self.exact_name_idx.get((country, name), []))
        if trans:
            candidates.update(self.trans_name_idx.get((country, trans), []))

        # 2. Relaxed Matching (Only if country is known)
        if country != "UNKNOWN":
            if name and len(name) >= 5:
                phonetic_key = jellyfish.metaphone(name)[:6]
                candidates.update(self.phonetic_idx.get((country, phonetic_key), []))
                
            if name:
                for token in set(name.split()):
                    if len(token) >= 4 and self.name_token_freq.get(token, 0) <= self.max_name_freq:
                        candidates.update(self.name_token_idx.get((country, token), []))
                        
            if addr:
                for token in set(addr.split()):
                    if len(token) >= 5 and self.addr_token_freq.get(token, 0) <= self.max_addr_freq:
                        candidates.update(self.addr_token_idx.get((country, token), []))
        
        # 3. Missing Country Fallback (Must be an exact string match, no loose tokens)
        else:
             if name:
                # Search across all known countries for this EXACT name
                for key_country, key_name in self.exact_name_idx.keys():
                    if key_name == name:
                        candidates.update(self.exact_name_idx.get((key_country, key_name), []))
             if trans:
                for key_country, key_trans in self.trans_name_idx.keys():
                    if key_trans == trans:
                        candidates.update(self.trans_name_idx.get((key_country, key_trans), []))

        return candidates

In [ ]:
import pandas as pd
import csv
import gc
import time
import os

# 1. Load only required columns
cols = ['entity_id', 'country', 'clean_name', 'Trans_name', 'norm_address']

print("Combining S2 and S3...")
target_df = pd.concat([s2[cols], s3[cols]], ignore_index=True)

# 2. Extract string IDs to Numpy Array
target_id_map = target_df['entity_id'].values 
target_df = target_df.drop(columns=['entity_id']) 

# 3. Aggressively clean up memory
del s2
del s3
gc.collect()

# 4. Initialize and fit the STRICT generator
print("Fitting the Strict Candidate Generator...")
start_fit_time = time.time()
generator = StrictCandidateGenerator(max_name_freq=100, max_addr_freq=50)
generator.fit(target_df)
print(f"Fitting complete in {time.time() - start_fit_time:.2f} seconds.")

del target_df
gc.collect()

from concurrent.futures import ProcessPoolExecutor, as_completed

# [Keep Steps 1-5 exactly the same: Load S2/S3, Build the Strict Generator]

def process_chunk(chunk_df, target_id_map, generator):
    """Function executed by parallel worker processes"""
    chunk_results = []
    for row in chunk_df.itertuples(index=False):
        row_dict = row._asdict()
        matched_int_indices = generator.get_candidates(row_dict)
        for match_idx in matched_int_indices:
            chunk_results.append([row.entity_id, target_id_map[match_idx]])
    return chunk_results

if __name__ == '__main__':
    output_file = "strict_s1_candidate_pairs.csv"
    print(f"Generating candidates in parallel...")
    start_gen_time = time.time()
    
    # Write header
    with open(output_file, mode='w', newline='') as f:
        csv.writer(f).writerow(["s1_entity_id", "target_entity_id"])
    
    # Split S1 into chunks of 10,000 rows
    chunk_size = 10000
    s1_chunks = [s1.iloc[i:i + chunk_size] for i in range(0, len(s1), chunk_size)]
    total_pairs = 0
    
    # Launch parallel workers (Adjust max_workers based on your CPU cores, e.g., 8 or 16)
    with ProcessPoolExecutor(max_workers=8) as executor:
        # Submit all chunks to the workers
        futures = {executor.submit(process_chunk, chunk, target_id_map, generator): chunk for chunk in s1_chunks}
        
        # As workers finish their chunks, append the results to disk
        for count, future in enumerate(as_completed(futures), 1):
            results = future.result()
            if results:
                with open(output_file, mode='a', newline='') as f:
                    csv.writer(f).writerows(results)
                total_pairs += len(results)
            
            # Print status every 5 chunks
            if count % 5 == 0:
                print(f"Processed {(count * chunk_size):,} / {len(s1):,} S1 entities | Pairs: {total_pairs:,}")

    print(f"Parallel generation complete in {time.time() - start_gen_time:.2f} seconds.")

In [ ]:
import pandas as pd
import duckdb
import jellyfish
import time
import gc
from concurrent.futures import ProcessPoolExecutor
import os

# Clean up any leftover database file from previous failed runs
if os.path.exists("pipeline.duckdb"):
    os.remove("pipeline.duckdb")

print("Initializing Disk-Backed DuckDB Engine...")
# 1. Use a disk-backed file instead of :memory: so it can spill safely to disk
con = duckdb.connect(database='pipeline.duckdb')

# 2. Allocate 60GB to DuckDB (leaving 68GB of RAM for OS and Pandas)
con.execute("PRAGMA threads=16")
con.execute("PRAGMA memory_limit='60GB'")
con.execute("PRAGMA temp_directory='duckdb_temp_buffer'")

# Register DataFrames (zero-copy)
con.register('s1', s1)
con.register('target', target_df)

start_sql = time.time()
print("Executing Multi-Pass Blocking with Frequency-Capped Phonetics...")

query = """
    -- Step A: Identify only RARE phonetic keys in the target dataset (<= 50 occurrences)
    -- This eliminates "heavy hitter" collisions (Bharat, Global, Trading, etc.)
    CREATE TEMP TABLE rare_target_phonetics AS
    SELECT country, phonetic
    FROM target
    WHERE phonetic IS NOT NULL AND country IS NOT NULL
    GROUP BY country, phonetic
    HAVING count(*) <= 50;

    -- Step B: Build deduplicated candidate pairs
    CREATE TABLE candidate_pairs AS
    
    -- Pass 1: Exact Clean Name
    SELECT s1.entity_id AS s1_id, t.entity_id AS target_id 
    FROM s1 
    INNER JOIN target t ON s1.clean_name = t.clean_name 
    WHERE s1.clean_name IS NOT NULL AND s1.clean_name != ''
    
    UNION
    
    -- Pass 2: Exact Transliterated Name
    SELECT s1.entity_id AS s1_id, t.entity_id AS target_id 
    FROM s1 
    INNER JOIN target t ON s1.Trans_name = t.Trans_name 
    WHERE s1.Trans_name IS NOT NULL AND s1.Trans_name != ''
    
    UNION
    
    -- Pass 3: Phonetic Match (Filtered by Rare Keys ONLY)
    SELECT s1.entity_id AS s1_id, t.entity_id AS target_id 
    FROM s1 
    INNER JOIN target t ON s1.phonetic = t.phonetic AND s1.country = t.country
    INNER JOIN rare_target_phonetics r ON t.phonetic = r.phonetic AND t.country = r.country
    WHERE s1.phonetic IS NOT NULL;
"""

con.execute(query)
print(f"Blocking query completed in {time.time() - start_sql:.2f} seconds.")

# Export directly to disk via C++ streaming
print("Streaming candidates to disk...")
start_export = time.time()
output_file = "duckdb_candidate_pairs.csv"
con.execute(f"COPY candidate_pairs TO '{output_file}' (HEADER, DELIMITER ',')")
print(f"Export completed in {time.time() - start_export:.2f} seconds.")

# Inspect final pair count
pair_count = con.execute("SELECT count(*) FROM candidate_pairs").fetchone()[0]
file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"Success! Generated {pair_count:,} candidate pairs ({file_size_mb:.2f} MB).")

# Close connection and cleanup temp buffer
con.close()

In [ ]:
data = pd.read_csv("duckdb_candidate_pairs.csv")

In [ ]:
data.sample(5)

In [ ]:
duck_match = list(data[data["s1_id"]=="S1-647273785"]['target_id'])
duck_match.sort()
duck_match

In [ ]:

import pandas as pd
gt = pd.read_csv('ML2026/student_resource/dataset/train/train_ground_truth.tsv', sep='\t')

In [ ]:
gt.sample(5)

In [ ]:
matched = gt[gt['source1_entity_id'] == 'S1-647273785']['matched_entity_ids']

In [ ]:
list(matched)

In [ ]:
for i in list(matched):
    if i in duck_match:
        print(i)

In [ ]:
#s2[s2['entity_id']=='S2-434801890']["business_address"]
s2.loc[2078092]["business_address"]

In [ ]:
s2.sample()

# approach 4: TF-IDF + Sparse Dot Top-N

In [ ]:
pip install scikit-learn sparse_dot_topn

In [ ]:
#CANDIDATE_GENERATION
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
import time
import gc
import csv
import pickle

def build_search_corpus(df):
    return (df['clean_name'].fillna('') + " " + 
            df['Trans_name'].fillna('') + " " + 
            df['norm_address'].fillna('')).str.lower()

# 1. Prepare Data
print("Preparing data...")
cols = ['entity_id', 'clean_name', 'Trans_name', 'norm_address']
target_df = pd.concat([s2[cols], s3[cols]], ignore_index=True)

s1_ids = s1['entity_id'].values
target_ids = target_df['entity_id'].values

s1_corpus = build_search_corpus(s1)
target_corpus = build_search_corpus(target_df)


    
del target_df, s2, s3
gc.collect()

# 2. Optimized Vectorization
print("Vectorizing text...")
start_vec = time.time()

vectorizer = TfidfVectorizer(
    analyzer='char_wb', 
    ngram_range=(3, 3), 
    min_df=5,            # Ignore typos (must appear at least 5 times)
    max_df=0.01,         # CRITICAL: Ignore generic text appearing in >5% of documents
    dtype=np.float32     # CRITICAL: 32-bit float for faster math
)

# Fit and transform
#vectorizer.fit(target_corpus)
print('vectorizing target_corpus')
#target_matrix = vectorizer.transform(target_corpus)
with open("target_matrix.pkl", "rb") as file:
    target_matrix = pickle.load(file)
print('vectorizing s1_corpus')
#s1_matrix = vectorizer.transform(s1_corpus)
with open("s1_corpus.pkl", "rb") as file:
    s1_matrix= pickle.load(file)



del s1_corpus, target_corpus
gc.collect()
print(f"Vectorization complete in {time.time() - start_vec:.2f} seconds.")

# 3. Chunked Sparse Matrix Multiplication
print("Executing Chunked Sparse Dot Product...")
start_math = time.time()
output_filename = "tfidf_candidate_pairs.csv"

# Pre-transpose the target matrix so we don't do it inside the loop
target_matrix_T = target_matrix.transpose()
chunk_size = 20000
total_pairs = 0

with open(output_filename, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["s1_entity_id", "target_entity_id", "tfidf_similarity"])
    
    for i in range(0, s1_matrix.shape[0], chunk_size):
        chunk_start_time = time.time()
        end = min(i + chunk_size, s1_matrix.shape[0])
        s1_chunk = s1_matrix[i:end]
        
        # Multiply only the 20k chunk against the full target dataset
        chunk_matches = sp_matmul_topn(
            s1_chunk, 
            target_matrix_T, 
            top_n=20, 
            threshold=0.60, 
            n_threads=140
        )
        
        # Extract indices
        non_zeros = chunk_matches.nonzero()
        
        # Adjust S1 row indices by the current chunk offset 'i'
        chunk_s1_indices = non_zeros[0] + i
        chunk_target_indices = non_zeros[1]
        chunk_scores = chunk_matches.data
        
        # Map back to string IDs
        matched_s1_ids = s1_ids[chunk_s1_indices]
        matched_target_ids = target_ids[chunk_target_indices]
        
        # Format for CSV writing
        batch_rows = zip(matched_s1_ids, matched_target_ids, np.round(chunk_scores, 4))
        writer.writerows(batch_rows)
        
        total_pairs += len(chunk_scores)
        print(f"Processed row {end:,} / {s1_matrix.shape[0]:,} "
              f"| Chunk Time: {time.time() - chunk_start_time:.1f}s "
              f"| Total Pairs: {total_pairs:,}")

print(f"Pipeline finished entirely in {time.time() - start_math:.2f} seconds.")

In [ ]:
!pip install jellyfish

In [ ]:
#TRAINING

import pandas as pd
import numpy as np
import jellyfish
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
import time
import gc

# ==========================================
# 1. PARALLEL FEATURE ENGINEERING WORKER
# ==========================================
def compute_features(df_chunk):
    # Suppress Pandas copy warnings since we are working on isolated chunks
    df_chunk = df_chunk.copy()
    
    n1 = df_chunk['norm_name_s1'].fillna('').astype(str).values
    n2 = df_chunk['norm_name_tgt'].fillna('').astype(str).values
    a1 = df_chunk['norm_address_s1'].fillna('').astype(str).values
    a2 = df_chunk['norm_address_tgt'].fillna('').astype(str).values
    c1 = df_chunk['country_s1'].fillna('').astype(str).values
    c2 = df_chunk['country_tgt'].fillna('').astype(str).values

    n_rows = len(df_chunk)
    jaro_name = np.zeros(n_rows, dtype=np.float32)
    lev_address = np.zeros(n_rows, dtype=np.float32)
    country_match = np.zeros(n_rows, dtype=np.int8)
    len_diff_name = np.zeros(n_rows, dtype=np.int32)

    for i in range(n_rows):
        s1_name, tgt_name = n1[i], n2[i]
        
        # 1. Jaro-Winkler
        jaro_name[i] = jellyfish.jaro_winkler_similarity(s1_name, tgt_name)
        
        # 2. Length Difference (Fixed: calculated natively in the loop)
        len_diff_name[i] = abs(len(s1_name) - len(tgt_name))
        
        # 3. Normalized Levenshtein for Address
        addr1, addr2 = a1[i], a2[i]
        max_len = max(len(addr1), len(addr2))
        if max_len > 0: 
            lev_address[i] = 1.0 - (jellyfish.levenshtein_distance(addr1, addr2) / max_len)
            
        # 4. Boolean Country Match
        if c1[i] == c2[i] and c1[i] != "": 
            country_match[i] = 1

    # Attach computed arrays to the dataframe chunk
    df_chunk['jaro_name'] = jaro_name
    df_chunk['lev_address'] = lev_address
    df_chunk['country_match'] = country_match
    df_chunk['len_diff_name'] = len_diff_name
    
    return df_chunk
# ==========================================
# 2. DATA LOADING & LABELING
# ==========================================
if __name__ == '__main__':
    print("Loading Candidates and Ground Truth...")
    candidates = pd.read_csv("candidate_pair_mthd4/tfidf_candidate_pairs.csv")
    ground_truth = pd.read_csv("ML2026/student_resource/dataset/train/train_ground_truth.tsv", sep="\t")
    target_df = pd.concat([s2,s3], ignore_index=True)

    # Generate True Positive pairs
    ground_truth['matched_entity_ids'] = ground_truth['matched_entity_ids'].fillna('').str.split(',')
    gt_exploded = ground_truth.explode('matched_entity_ids')
    gt_exploded = gt_exploded[gt_exploded['matched_entity_ids'] != '']
    true_pairs = set(zip(gt_exploded['source1_entity_id'], gt_exploded['matched_entity_ids']))

    # Vectorized Labeling
    print("Assigning Labels...")
    candidate_tuples = list(zip(candidates['s1_entity_id'], candidates['target_entity_id']))
    candidates['label'] = [1 if pair in true_pairs else 0 for pair in candidate_tuples]
    del true_pairs, candidate_tuples, ground_truth, gt_exploded
    gc.collect()

    print("Merging Text Fields...")
    # Assuming s1 and target_df are loaded in memory
    df = candidates.merge(s1[['entity_id', 'norm_name', 'norm_address', 'country']], 
                          left_on='s1_entity_id', right_on='entity_id', how='left')
    df = df.merge(target_df[['entity_id', 'norm_name', 'norm_address', 'country']], 
                  left_on='target_entity_id', right_on='entity_id', suffixes=('_s1', '_tgt'), how='left')

    # ==========================================
# 3. MASSIVELY PARALLEL EXECUTION
    # ==========================================
    print("Executing 148-Core Feature Engineering...")
    start_feat = time.time()
    
    # Split the DataFrame into roughly 600 chunks (4 chunks per core). 
    # This prevents cores from sitting idle if one chunk has longer strings than another.
    num_cores = 140 
    df_chunks = np.array_split(df, num_cores * 4)

    # Distribute the chunks across the cores
    with ProcessPoolExecutor(max_workers=num_cores) as executor:
        processed_chunks = list(executor.map(compute_features, df_chunks))

    # Stitch the dataset back together
    df = pd.concat(processed_chunks, ignore_index=True)
    del df_chunks, processed_chunks
    gc.collect()
    print(f"Feature Engineering complete in {time.time() - start_feat:.2f} seconds.")

    # ==========================================
    # 4. LIGHTGBM HYPER-THREADED TRAINING
    # ==========================================
    features = ['tfidf_similarity', 'jaro_name', 'lev_address', 'country_match', 'len_diff_name']

    # Grouped Split to prevent data leakage
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, val_idx = next(gss.split(df, groups=df['s1_entity_id']))

    train_data = df.iloc[train_idx]
    val_data = df.iloc[val_idx].copy()

    X_train, y_train = train_data[features], train_data['label']
    X_val, y_val = val_data[features], val_data['label']

    print("\nTraining LightGBM on 140 Threads...")
    # n_jobs natively maps LightGBM tree construction to your specific core count
    model = lgb.LGBMClassifier(
        n_estimators=1000,          # Increased trees since training is so fast now
        learning_rate=0.05, 
        is_unbalance=True, 
        n_jobs=140,                 # UNLEASH THE CORES
        random_state=42
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50)]
    )

    # ==========================================
    # 5. F0.5 THRESHOLD CALIBRATION
    # ==========================================
    print("\nEvaluating F0.5 Metric:")
    val_data['match_prob'] = model.predict_proba(X_val)[:, 1]

    best_f05 = 0
    best_thresh = 0.5

    for thresh in np.arange(0.70, 0.96, 0.05):
        val_data['pred'] = (val_data['match_prob'] >= thresh).astype(int)
        
        tp = len(val_data[(val_data['pred'] == 1) & (val_data['label'] == 1)])
        fp = len(val_data[(val_data['pred'] == 1) & (val_data['label'] == 0)])
        fn = len(val_data[(val_data['pred'] == 0) & (val_data['label'] == 1)])
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f05 = (1.25 * precision * recall) / ((0.25 * precision) + recall) if (precision + recall) > 0 else 0
            
        print(f"Threshold: {thresh:.2f} | Precision: {precision:.3f} | Recall: {recall:.3f} | F0.5: {f05:.3f}")
        
        if f05 > best_f05:
            best_f05 = f05
            best_thresh = thresh

    print(f"\nOptimal F0.5 Threshold: {best_thresh:.2f}")

In [ ]:
#TESTING
import joblib
import json

print("Saving models to disk...")

# 1. Save the LightGBM Classifier
# joblib heavily compresses the tree structures, making it much smaller than standard pickle
joblib.dump(model, 'lgbm_f05_model.pkl')

# 2. Save the TF-IDF Vectorizer
# This saves the 500,000 exact n-gram vocabulary mappings
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

# 3. Save your Optimal Threshold
# Storing it in a simple JSON so you don't forget the exact F0.5 cutoff
with open('pipeline_config.json', 'w') as f:
    json.dump({'best_f05_threshold': best_thresh}, f)

print("Pipeline successfully saved! You can now close the kernel.")

In [2]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sparse_dot_topn import sp_matmul_topn
import jellyfish
import lightgbm as lgb
from concurrent.futures import ProcessPoolExecutor
from anyascii import anyascii
import joblib
import json
import time
import re
import gc
import os

# ==============================================================================
# 1. COMPILED EDA CLEANING & TRANSLITERATION LOGIC
# ==============================================================================
RAW_MULTILINGUAL_SEEDS = {
    'private': [
        'प्राइवेट', 'प्रायव्हेट', 'ప్రైవేట్', 'பிரைவேட்', 'প্রাইভেট', 'પ્રાઇવેટ', 
        'ಖಾಸಗಿ', 'പ്രൈവറ്റ്', 'ਪ੍ਰਾਈਵੇਟ', 'ପ୍ରାଇଭେଟ', 'پرائیویٹ', 'privé', 'privée'
    ],
    'limited': [
        'लिमिटेड', 'मर्यादित', 'లిమిటెడ్', 'லிமிடெட்', 'লিমিটেড', 'લિમિટેડ', 
        'ಲಿಮಿಟೆಡ್', 'ലിമിറ്റഡ്', 'ਲਿਮਿਟੇਡ', 'ଲିମିଟେଡ', 'لمیٹڈ', 'limitée', 'limité'
    ],
    'company': [
        'कंपनी', 'కంపెనీ', 'கம்பெனி', 'কোম্পানি', 'કંપની', 'ಕಂಪನಿ', 
        'കമ്പനി', 'ਕੰਪਨੀ', 'କମ୍ପାନୀ', 'کمپنی', 'société', 'compagnie'
    ],
    'corporation': [
        'कॉर्पोरेशन', 'निगम', 'కార్పొరేషన్', 'கார்ப்பரேஷன்', 'কর্পোরেশন', 'કોર્પોરેશન', 
        'ಕಾರ್ಪೊರೇಷನ್', 'കോർപ്പറേഷൻ', 'ਕਾਰਪੋਰੇਸ਼ਨ', 'କର୍ପୋରେସନ', 'کارپوریشن'
    ],
    'incorporated': [
        'इनकॉर्पोरेटेड', 'ఇన్కార్పొరేటెڈ', 'இன்கார்பரேட்டட்', 'ইনকর্পোরেটেড', 
        'ઇનકોર્પોરેટેડ', 'ಇನ್ಕಾರ್ಪೊರೇಟೆಡ್', 'ഇൻകോർപ്പറേറ്റഡ്', 'ਇਨਕਾਰਪੋਰੇਟਿਡ', 'ଇନକର୍ପୋରେଟେਡ'
    ],
    'llp': [
        'एलएलपी', 'ఎల్‌ఎల్‌పి', 'எல்.எல்.பி', 'এলএলপি', 'એલએલપી', 'ಎಲ್ಎಲ್ಪಿ', 
        'എൽഎൽപി', 'ਐਲਐਲਪੀ', 'ଏଲଏଲପି'
    ],
    'finance': [
        'फाइनेंस', 'फायनान्स', 'ఫైనాన్స్', 'பைனான்ஸ்', 'ফাইন্যান্স', 'ફાઇનાન્સ', 
        'ಫೈನಾನ್ಸ್', 'ഫിനാൻസ്', 'ਫਾਈਨਾਂਸ', 'ଫାଇନାନ୍ସ', 'فنانس', 'finance'
    ],
    'bank': [
        'बैंक', 'बँक', 'బ్యాంక్', 'வங்கி', 'ব্যাঙ্ক', 'બેંક', 'ಬ್ಯಾಂಕ್', 'ബാങ്ക്', 
        'ਬੈਂਕ', 'ବ୍ୟାଙ୍କ', 'بینک', 'banque'
    ],
    'insurance': [
        'इंश्योरेंस', 'विमा', 'ఇన్సూరెన్స్', 'இன்சூரன்ஸ்', 'ইন্স্যুরেন্স', 'ઇન્શ્યોરન્સ', 
        'ಇನ್ಶೂರೆನ್ಸ್', 'ഇൻഷുറൻസ്', 'ਇੰਸ਼ੋਰੈਂਸ', 'ଇନ୍ସୁରାନ୍ସ', 'انشورنس', 'assurance'
    ],
    'investment': [
        'इन्वेस्टमेंट', 'गुंतवणूक', 'ఇన్వెస్ట్‌మెంట్', 'இன்வெஸ்ட்மென்ட்', 'ইনভেস্টমেন্ট', 
        'ઇન્વેસ્ટમેન્ટ', 'ಇನ್ವೆಸ್ಟ್ಮೆಂಟ್', 'ഇൻവെസ്റ്റ്മെന്റ്', 'ਇਨਵੈਸਟਮੈਂਟ', 'ଇନଭେଷ୍ଟମେଣ୍ଟ', 
        'انویسٹمنٹ', 'investissement'
    ],
    'technologies': [
        'टेक्नोलॉजीज', 'तंत्रज्ञान', 'టెక్నాలజీస్', 'டெக்னாலஜிஸ்', 'টেকনোলজিস', 
        'ટેકનોલોજીસ', 'ಟೆಕ್ನಾಲಜೀಸ್', 'ടെക്നോളജീസ്', 'ਟੈਕਨਾਲੋਜੀਜ਼', 'ଟେକ୍ନୋଲୋଜିସ', 'ٹیکنالوجیز'
    ],
    'consultancy': [
        'कंसल्टेंसी', 'పరామर्श', 'కన్సల్టెన్సీ', 'கன்சல்டன்சி', 'কনসালটেন্সি', 
        'કન્સલ્ટન્સી', 'ಕನ್ಸಲ್ಟೆನ್ಸಿ', 'കൺസൾട്ടൻസി', 'ਕੰਸਲਟੈਂਸੀ', 'କନସଲଟାନ୍ସି', 
        'کنسلٹنسی', 'consultance', 'conseil'
    ],
    'pharmacy': [
        'फार्मेसी', 'औषधालय', 'ఫార్మసీ', 'பார்மசி', 'ফার্মেসি', 'ફાર્મસી', 
        'ಫಾರ್ಮಸಿ', 'ഫാർമസി', 'ਫਾਰਮੇਸੀ', 'ଫାର୍ମାସି', 'فارمیسی', 'pharmacie'
    ],
    'logistics': [
        'लॉजिस्टिक्स', 'లాజిస్టిక్స్', 'லாஜிஸ்டிக்ஸ்', 'লজিস্টিকস', 'લોજિસ્ટિક્સ', 
        'ಲಾಜಿಸ್ಟಿಕ್ಸ್', 'ലോജിസ്റ്റിക്സ്', 'ਲੌਜਿਸਟਿਕਸ', 'ଲଜିଷ୍ଟିକ୍ସ', 'لاجسٹکس', 'logistique'
    ],
    'manufacturing': [
        'मैन्युफैक्चरिंग', 'उत्पादन', 'మాన్యుఫ్యాక్చరింగ్', 'மேனுஃபேக்ச்சரிங்', 
        'ম্যানুফ্যাকচারিং', 'મેન્યુફેક્ચરિંગ', 'ಮ್ಯಾನುಫ್ಯಾಕ್ಚರಿಂಗ್', 'മാനുഫാക്ചറിംഗ്', 
        'ਮੈਨੂਫੈਕਚਰਿੰਗ', 'ମାନୁଫାକ୍ଚରିଂ', 'مینوفیکچرنگ', 'fabrication'
    ],
    'group': [
        'ग्रुप', 'समूह', 'గ్రూప్', 'குரூப்', 'গ্রুপ', 'ગ્રુપ', 'ಗ್ರೂಪ್', 'ഗ്രൂപ്പ്', 
        'ਗਰੁੱਪ', 'ଗ୍ରୁପ', 'گروپ', 'groupe'
    ],
    'associates': [
        'एसोसिएट्स', 'असोसिएट्स', 'అసోసియేట్స్', 'அசோசியேட்ஸ்', 'অ্যাসোসিয়েটস', 
        'એસોસિએટ્સ', 'ಅಸೋಸಿಯೇಟ್ಸ್', 'അസോസിയേറ്റ്സ്', 'ਐਸੋਸੀਏਟਸ', 'ଆସୋସିଏଟ୍ସ', 'ایسوسی ایٹس', 'associés'
    ],
    'enterprises': [
        'एंटरप्राइजेज', 'उद्यम', 'ఎంటర్‌ప్రైజెస్', 'என்டர்பிரைசஸ்', 'এন্টারপ্রাইজ', 
        'એન્ટરપ્રાઇઝિસ', 'ಎಂಟರ್‌ಪ್ರೈಸಸ್', 'എന്റർപ്രൈസസ്', 'ਐਂਟਰਪ੍ਰਾਈਜ਼ਿਜ਼', 
        'ଏଣ୍ଟରପ୍ରାଇଜେସ', 'انٹرپرائزز', 'entreprises'
    ],
    'services': [
        'सर्विसेज', 'सेवा', 'సర్వీసెస్', 'சர்வீசஸ்', 'সার্ভিসেস', 'સર્વિસિસ', 
        'ಸರ್ವಿಸಸ್', 'സർവീസസ്', 'ਸਰਵਿਸਿਜ਼', 'ସର୍ଭିସେସ', 'سروسز', 'services'
    ],
    'solutions': [
        'सॉल्यूशंस', 'సొల్యూషన్స్', 'சொல்யூஷன்ஸ்', 'সলিউশন', 'સોલ્યુશન્સ', 
        'ಸಲ್ಯೂಷನ್ಸ್', 'സൊല്യൂഷൻസ്', 'ਸਲਿਊਸ਼ਨਜ਼', 'ସଲ୍ୟୁସନ', 'سلوشنز'
    ],
    'industries': [
        'इंडस्ट्रीज', 'उद्योग', 'ఇండస్ట్రీస్', 'இண்டஸ்ட்ரீஸ்', 'ইন্ডাস্ট্রিজ', 
        'ઇન્ડસ્ટ્રીઝ', 'ಇಂಡಸ್ಟ್ರೀಸ್', 'ഇൻഡസ്ട്രീസ്', 'ਇੰਡਸਟਰੀਜ਼', 'ଇଣ୍ଡଷ୍ଟ୍ରିଜ', 'انڈسٹریز'
    ],
    'global': [
        'ग्लोबल', 'जागतिक', 'గ్లోబల్', 'குளோபல்', 'গ্লোবাল', 'ગ્લોબલ', 'ಗ್ಲೋಬಲ್', 
        'ഗ്ലോബൽ', 'ਗਲੋਬਲ', 'ଗ୍ଲୋବାଲ', 'گلوبل', 'mondial'
    ],
    'ventures': [
        'वेंचर्स', 'వెంచర్స్', 'வென்சர்ஸ்', 'ভেঞ্চারস', 'વેન્ચર્સ', 'ವೆಂಚರ್ಸ್', 
        'വെഞ്ചേഴ്സ്', 'ਵੈਂਚਰਸ', 'ଭେଞ୍ଚର୍ସ', 'وینچرز'
    ],
    'Rajasthan': ['राजस्थान', 'రాజస్థాన్', 'ராஜஸ்தான்', 'রাজস্থান', 'રાજસ્થાન', 'ਰਾਜਸਥਾਨ'],
    'Maharashtra': ['महाराष्ट्र', 'మహారాష్ట్ర', 'மகாராஷ்டிரா', 'মহারাষ্ট্র', 'મહારાષ્ટ્ર'],
    'Gujarat': ['गुजरात', 'గుజరాత్', 'குஜராத்', 'ગુજરાત', 'ગુજરાત'],
    'Punjab': ['पंजाब', 'పంజాబ్', 'பஞ்சாப்', 'পাঞ্জাব', 'પંજાબ', 'ਪੰਜਾਬ'],
    'Uttar Pradesh': ['उत्तर प्रदेश', 'ఉత్తర ప్రదేశ్', 'உத்தர பிரதேசம்', 'উত্তর প্রদেশ', 'ઉત્તર પ્રદેશ'],
    'Madhya Pradesh': ['मध्य प्रदेश', 'మధ్య ప్రదేశ్', 'மத்திய பிரதேசம்', 'মধ্য প্রদেশ', 'મધ્ય પ્રદેશ'],
    'Karnataka': ['कर्नाटक', 'కర్ణాటక', 'கர்நாடகா', 'কর্ণাটক', 'કર્ણાટક', 'ಕರ್ನಾಟಕ'],
    'Tamil Nadu': ['तमिलनाडु', 'తమిళనాడు', 'தமிழ்நாடு', 'তামিলনাড়ু', 'તમિલનાડુ'],
    'Kerala': ['केरल', 'కేరళ', 'கேரளா', 'কেরালা', 'કેરળ', 'കേരളം'],
    'West Bengal': ['पश्चिम बंगाल', 'పశ్చిమ బెంగాల్', 'மேற்கு வங்காளம்', 'পশ্চিমবঙ্গ', 'પશ્ચિમ બંગાળ'],
    'Bihar': ['बिहार', 'బీహార్', 'பீகார்', 'বিহার', 'બિહાર'],
    'Odisha': ['ओडिशा', 'ఒడిశా', 'ஒடிசா', 'ওড়িশা', 'ઓડિશા', 'ଓଡ଼ିଶା'],
    'Andhra Pradesh': ['आंध्र प्रदेश', 'ఆంధ్రప్రదేశ్', 'ஆந்திர பிரதேசம்', 'অন্ধ্র প্রদেশ', 'આંધ્ર પ્રદેશ'],
    'Telangana': ['तेलंगाना', 'తెలంగాణ', 'தெலுங்கானா', 'তেলেঙ্গானா', 'તેલંગાણા'],
    'Assam': ['असम', 'అస్సాం', 'அசாம்', 'আসাম', 'અસમ'],
    'Haryana': ['हरियाणा', 'హర్యానా', 'ஹரியானா', 'হরিয়ানা', 'હરિયાણા', 'ਹਰਿਆਣਾ'],
    'Delhi': ['दिल्ली', 'ఢిల్లీ', 'டெல்லி', 'দিল্লি', 'દિલ્હી', 'ਦਿੱਲੀ']
}

def build_compiled_phonetic_dictionary(seed_dict):
    compiled_patterns = {}
    for canonical_english, native_terms in seed_dict.items():
        ascii_variants = set()
        for term in native_terms:
            ascii_repr = anyascii(term).lower().strip()
            ascii_repr = re.sub(r'[^\w\s]', '', ascii_repr)
            if ascii_repr:
                ascii_variants.add(ascii_repr)
        sorted_variants = sorted(list(ascii_variants), key=len, reverse=True)
        pattern_str = r'\b(' + '|'.join(map(re.escape, sorted_variants)) + r')\b'
        compiled_patterns[re.compile(pattern_str)] = canonical_english
    return compiled_patterns

PHONETIC_REGEX_MAPPING = build_compiled_phonetic_dictionary(RAW_MULTILINGUAL_SEEDS)

NAME_SUFFIX_MAPPING = {
    re.compile(r'\bcorp\b'): 'corporation',
    re.compile(r'\bpvt\b'): 'private',
    re.compile(r'\bltd\b'): 'limited',
    re.compile(r'\binc\b'): 'incorporated',
    re.compile(r'\bllc\b'): 'limited liability company',
    re.compile(r'\bco\b'): 'company',
    re.compile(r'\bpc\b'): 'professional corporation'
}

ADDR_MAPPING = {
    re.compile(r'\brd\b'): 'road',
    re.compile(r'\bst\b'): 'street',
    re.compile(r'\bave\b'): 'avenue',
    re.compile(r'\bblvd\b'): 'boulevard',
    re.compile(r'\bbldg\b'): 'building',
    re.compile(r'\bapt\b'): 'apartment',
    re.compile(r'\bste\b'): 'suite',
    re.compile(r'\bfl\b'): 'floor',
    re.compile(r'\bno\b'): 'number'
}

RE_LANDMARK = re.compile(r'\b(near|opp|opposite)\s+\w+(?:\s+\w+){0,3}')
RE_NULL = re.compile(r'(\b(na|null|nan|nill)\b)')
RE_PUNCT = re.compile(r'[^\w\s]')

def std_bus_name(name):
    if pd.isna(name):
        return ""
    name = str(name).lower().replace("&", " and ")
    name = RE_PUNCT.sub('', name)
    for pattern, replacement in NAME_SUFFIX_MAPPING.items():
        name = pattern.sub(replacement, name)
    return " ".join(name.split())

def norm_add(address):
    if pd.isna(address):
        return ""
    address = str(address).lower()
    address = RE_PUNCT.sub('', address)
    for pattern, replacement in ADDR_MAPPING.items():
        address = pattern.sub(replacement, address)
    address = RE_LANDMARK.sub(' ', address)
    address = RE_NULL.sub('', address)
    return " ".join(address.split())

def clean_name_data(norm_name_str, norm_addr_str):
    x_tokens = [tok for tok in norm_name_str.split() if not tok.isdigit()]
    y_tokens = set(norm_addr_str.split())
    final_tokens = [token for token in x_tokens if token not in y_tokens]
    return " ".join(final_tokens)

def normalize_text_pipeline(text):
    if pd.isna(text) or not text:
        return ""
    text_ascii = anyascii(str(text)).lower()
    text_ascii = RE_PUNCT.sub(' ', text_ascii)
    for pattern, canonical_term in PHONETIC_REGEX_MAPPING.items():
        text_ascii = pattern.sub(canonical_term, text_ascii)
    return " ".join(text_ascii.split())

# Parallel Preprocessing Worker
def process_dataframe_chunk(df_chunk):
    df_chunk = df_chunk.copy()
    
    # 1. Standardize names & addresses
    norm_names = [std_bus_name(x) for x in df_chunk['business_name'].values]
    norm_addrs = [norm_add(x) for x in df_chunk['business_address'].values]
    
    # 2. Strip address tokens from name
    clean_names = [clean_name_data(n, a) for n, a in zip(norm_names, norm_addrs)]
    
    # 3. Transliterate clean name & address
    trans_names = [normalize_text_pipeline(cn) for cn in clean_names]
    trans_addrs = [normalize_text_pipeline(na) for na in norm_addrs]
    
    df_chunk['norm_name'] = norm_names
    df_chunk['norm_address'] = norm_addrs
    df_chunk['clean_name'] = clean_names
    df_chunk['Trans_name'] = trans_names
    df_chunk['Trans_address'] = trans_addrs
    return df_chunk

def parallel_preprocess(df, num_workers=140):
    chunks = np.array_split(df, num_workers * 2)
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        processed_chunks = list(executor.map(process_dataframe_chunk, chunks))
    return pd.concat(processed_chunks, ignore_index=True)


# ==============================================================================
# 2. PIPELINE EXECUTION (TEST SET)
# ==============================================================================
if __name__ == '__main__':
    print("Loading Trained Pipeline Assets...")
    model = joblib.load('lgbm_f05_model.pkl')

    with open('pipeline_config.json', 'r') as f:
        best_thresh = json.load(f)['best_f05_threshold']

    print(f"Optimal F0.5 Threshold Loaded: {best_thresh:.2f}")

    print("\nReading Raw TSV Test Files...")
    test_s1 = pd.read_csv("ML2026/student_resource/dataset/test/test_source1.tsv", sep='\t')
    test_s2 = pd.read_csv("ML2026/student_resource/dataset/test/test_source2.tsv", sep='\t')
    test_s3 = pd.read_csv("ML2026/student_resource/dataset/test/test_source3.tsv", sep='\t')

    print("Executing 148-Core Parallel Preprocessing...")
    start_eda = time.time()
    test_s1 = parallel_preprocess(test_s1, num_workers=140)
    test_s2 = parallel_preprocess(test_s2, num_workers=140)
    test_s3 = parallel_preprocess(test_s3, num_workers=140)
    print(f"Preprocessing completed in {time.time() - start_eda:.2f} seconds.")

    # Combine S2 & S3 Targets
    cols = ['entity_id', 'norm_name', 'norm_address', 'country', 'clean_name', 'Trans_name']
    target_df = pd.concat([test_s2[cols], test_s3[cols]], ignore_index=True)

    test_ids = test_s1['entity_id'].values
    target_ids = target_df['entity_id'].values

    def build_search_corpus(df):
        return (df['clean_name'].fillna('') + " " + 
                df['Trans_name'].fillna('') + " " + 
                df['norm_address'].fillna('')).str.lower()

    from sklearn.feature_extraction.text import TfidfVectorizer
    
    # ==============================================================================
    # 3. TF-IDF CANDIDATE GENERATION (Re-fitted on Test Data)
    # ==============================================================================
    print("\nRebuilding and Fitting Vectorizer on Test Target Data...")
    start_vec = time.time()
    
    # Recreate the exact strict vectorizer used in training
    vectorizer = TfidfVectorizer(
        analyzer='char_wb', 
        ngram_range=(3, 3), 
        min_df=10,               
        max_df=0.01,             
        max_features=500000,     
        dtype=np.float32         
    )
    
    target_corpus = build_search_corpus(target_df)
    test_corpus = build_search_corpus(test_s1)
    
    # Fit the vocabulary strictly on the Test Targets (S2 + S3)
    print("Fitting vocabulary dictionary...")
    vectorizer.fit(target_corpus)
    
    print("Transforming text to mathematical matrices...")
    target_matrix = vectorizer.transform(target_corpus)
    test_matrix = vectorizer.transform(test_corpus)
    target_matrix_T = target_matrix.transpose()
    
    # Free up the raw string arrays from RAM
    del target_corpus, test_corpus
    gc.collect()
    
    print(f"Vectorization completed entirely in {time.time() - start_vec:.2f} seconds.")
    
    print("Running 148-Core Top-N Sparse Dot Product...")
    chunk_size = 50000
    test_cand_s1 = []
    test_cand_tgt = []
    test_cand_sim = []
    
    for i in range(0, test_matrix.shape[0], chunk_size):
        end = min(i + chunk_size, test_matrix.shape[0])
        chunk_matches = sp_matmul_topn(
            test_matrix[i:end], 
            target_matrix_T, 
            top_n=20, 
            threshold=0.60, 
            n_threads=140
        )
        non_zeros = chunk_matches.nonzero()
        test_cand_s1.extend(test_ids[non_zeros[0] + i])
        test_cand_tgt.extend(target_ids[non_zeros[1]])
        test_cand_sim.extend(chunk_matches.data)
    
    candidates = pd.DataFrame({
        's1_entity_id': test_cand_s1,
        'target_entity_id': test_cand_tgt,
        'tfidf_similarity': test_cand_sim
    })
    
    del test_matrix, target_matrix, target_matrix_T, test_cand_s1, test_cand_tgt, test_cand_sim
    gc.collect()
    # ==============================================================================
    # 4. EXPORT MANDATORY candidate_pairs.tsv
    # ==============================================================================
    print("\nFormatting 'candidate_pairs.tsv' for competition validator...")
    os.makedirs("output", exist_ok=True)

    raw_candidates = candidates.groupby('s1_entity_id')['target_entity_id'].apply(
        lambda x: ','.join(list(dict.fromkeys(x)))
    ).reset_index()
    raw_candidates.rename(columns={'s1_entity_id': 'source1_entity_id', 'target_entity_id': 'candidate_entity_ids'}, inplace=True)

    all_test_ids = pd.DataFrame({'source1_entity_id': test_ids})
    candidate_submission = all_test_ids.merge(raw_candidates, on='source1_entity_id', how='left')
    candidate_submission['candidate_entity_ids'] = candidate_submission['candidate_entity_ids'].fillna('')
    candidate_submission.to_csv("output/candidate_pairs.tsv", sep='\t', index=False)
    print("Exported: output/candidate_pairs.tsv")

    # ==============================================================================
    # 5. PARALLEL FEATURE ENGINEERING
    # ==============================================================================
    print("\nMerging metadata for feature extraction...")
    df = candidates.merge(test_s1[['entity_id', 'norm_name', 'norm_address', 'country']], 
                          left_on='s1_entity_id', right_on='entity_id', how='left')
    df = df.merge(target_df[['entity_id', 'norm_name', 'norm_address', 'country']], 
                  left_on='target_entity_id', right_on='entity_id', suffixes=('_s1', '_tgt'), how='left')

    def compute_features(df_chunk):
        df_chunk = df_chunk.copy()
        n1 = df_chunk['norm_name_s1'].fillna('').astype(str).values
        n2 = df_chunk['norm_name_tgt'].fillna('').astype(str).values
        a1 = df_chunk['norm_address_s1'].fillna('').astype(str).values
        a2 = df_chunk['norm_address_tgt'].fillna('').astype(str).values
        c1 = df_chunk['country_s1'].fillna('').astype(str).values
        c2 = df_chunk['country_tgt'].fillna('').astype(str).values

        n_rows = len(df_chunk)
        jaro_name = np.zeros(n_rows, dtype=np.float32)
        lev_address = np.zeros(n_rows, dtype=np.float32)
        country_match = np.zeros(n_rows, dtype=np.int8)
        len_diff_name = np.zeros(n_rows, dtype=np.int32)

        for i in range(n_rows):
            s1_n, tgt_n = n1[i], n2[i]
            jaro_name[i] = jellyfish.jaro_winkler_similarity(s1_n, tgt_n)
            len_diff_name[i] = abs(len(s1_n) - len(tgt_n))
            
            addr1, addr2 = a1[i], a2[i]
            max_len = max(len(addr1), len(addr2))
            if max_len > 0:
                lev_address[i] = 1.0 - (jellyfish.levenshtein_distance(addr1, addr2) / max_len)
            
            if c1[i] == c2[i] and c1[i] != "":
                country_match[i] = 1

        df_chunk['jaro_name'] = jaro_name
        df_chunk['lev_address'] = lev_address
        df_chunk['country_match'] = country_match
        df_chunk['len_diff_name'] = len_diff_name
        return df_chunk

    print("Computing String Distance Features on 140 threads...")
    start_feat = time.time()
    df_chunks = np.array_split(df, 140 * 4)
    with ProcessPoolExecutor(max_workers=140) as executor:
        df = pd.concat(list(executor.map(compute_features, df_chunks)), ignore_index=True)
    print(f"Features engineered in {time.time() - start_feat:.2f} seconds.")

    # ==============================================================================
    # 6. LIGHTGBM INFERENCE & matching_results.tsv
    # ==============================================================================
    print("\nRunning LightGBM Inference...")
    features = ['tfidf_similarity', 'jaro_name', 'lev_address', 'country_match', 'len_diff_name']
    df['match_prob'] = model.predict_proba(df[features])[:, 1]
    df['is_match'] = (df['match_prob'] >= best_thresh).astype(int)

    print("Formatting 'matching_results.tsv'...")
    final_matches = df[df['is_match'] == 1]

    submission = final_matches.groupby('s1_entity_id')['target_entity_id'].apply(
        lambda x: ','.join(list(dict.fromkeys(x)))
    ).reset_index()
    submission.rename(columns={'s1_entity_id': 'source1_entity_id', 'target_entity_id': 'matched_entity_ids'}, inplace=True)

    submission_final = all_test_ids.merge(submission, on='source1_entity_id', how='left')
    submission_final['matched_entity_ids'] = submission_final['matched_entity_ids'].fillna('')
    submission_final.to_csv("output/matching_results.tsv", sep='\t', index=False)

    print("\nSUCCESS: Generated valid competition outputs in 'output/':")
    print(f" - output/matching_results.tsv ({len(submission_final):,} rows)")
    print(f" - output/candidate_pairs.tsv ({len(candidate_submission):,} rows)")

Loading Trained Pipeline Assets...
Optimal F0.5 Threshold Loaded: 0.90

Reading Raw TSV Test Files...
Executing 148-Core Parallel Preprocessing...


/opt/conda/lib/python3.12/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
/opt/conda/lib/python3.12/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
/opt/conda/lib/python3.12/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Preprocessing completed in 32.91 seconds.

Rebuilding and Fitting Vectorizer on Test Target Data...
Fitting vocabulary dictionary...
Transforming text to mathematical matrices...
Vectorization completed entirely in 388.41 seconds.
Running 148-Core Top-N Sparse Dot Product...

Formatting 'candidate_pairs.tsv' for competition validator...
Exported: output/candidate_pairs.tsv

Merging metadata for feature extraction...
Computing String Distance Features on 140 threads...


/opt/conda/lib/python3.12/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Features engineered in 91.04 seconds.

Running LightGBM Inference...
Formatting 'matching_results.tsv'...

SUCCESS: Generated valid competition outputs in 'output/':
 - output/matching_results.tsv (1,732,544 rows)
 - output/candidate_pairs.tsv (1,732,544 rows)


In [1]:
!pip install anyascii